系统提示词 通过create_agetn（）方法的system_prompt参数来设置，定义它的行为，参数值可以是string类型或者SystemMessage类型

In [2]:
from typing import Literal

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os

from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool


load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#1 初始化大模型
myModel= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)


#2 声明1个工具
class WeatherInput(BaseModel):
    """
    查询天气工具的形参的类的相关信息
    """
    city: str=Field(description="城市名称，例如'北京市','重庆市'",default="重庆市")
    unit:Literal["celsius","kelvin"]=Field(description="描述气温的单位")
    date:str=Field(description="查询日期")

@tool(description="查询具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:Literal["celsius","kelvin"],date:str):
    return f"{city} {date}的天气晴朗，气温10-16{unit}，有微风，适宜出行"
#3 初始化我的agent
myAgent=create_agent(
    model=myModel,
    tools=[get_weather],
    name="agent01",
    # system_prompt="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。"
    system_prompt=SystemMessage(content="你是1个专业的天气预报小助手,当有人问你关于天气问题的时候，你会调用工具get_weather来帮你完成查询任务。")
)
#4 调用agent
response=myAgent.invoke(
    {
        # 严格校验，使用message少了1个s就会报错

        "messages":
            [
                HumanMessage("请你查询一下重庆2026-9-15的天气")
        ]
    }
)
rprint(response)
# print(response)

{
    'messages': [
        HumanMessage(
            content='请你查询一下重庆2026-9-15的天气',
            additional_kwargs={},
            response_metadata={},
            id='f6d0538e-f16e-4ccf-8a4c-3ec5d08449d9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 119,
                    'prompt_tokens': 412,
                    'total_tokens': 531,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 58,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 119
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 412
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-a0ef1b3c-e7d6-9186-9c64-dfa9b23451c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0a921-488d-7a60-a9e6-711e215d331f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '重庆市', 'date': '2026-09-15', 'unit': 'celsius'},
                    'id': 'call_edaf8534e03b4f56aa47951b',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 412,
                'output_tokens': 119,
                'total_tokens': 531,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 58}
            }
        ),
        ToolMessage(
            content='重庆市 2026-09-15的天气晴朗，气温10-16celsius，有微风，适宜出行',
            name='get_weather',
            id='05cea2dd-1687-4029-961b-27442899b834',
            tool_call_id='call_edaf8534e03b4f56aa47951b'
        ),
        AIMessage(
            content='已为您查询到重庆市 2026年9月15日 的天气情况：\n\n🌤️ **重庆市 · 2026-09-15**\n- 
**天气状况**：晴朗\n- **气温**：10℃ ~ 16℃\n- **风力**：微风\n- **出行建议**：适宜出行 ☀️\n\n📌 
小提示：该日期距离现在较远，属于长期预报，实际天气可能会有较大变化，建议临近日期再次查询确认。另外，早晚气温偏低（
约10℃），出行可备一件薄外套。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 178,
                    'prompt_tokens': 517,
                    'total_tokens': 695,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 45,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 178
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': 517
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.8-max',
                'system_fingerprint': None,
                'id': 'chatcmpl-1296b785-125f-911b-8bef-33fecaafc724',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agent01',
            id='lc_run--01a0a921-54be-74c2-b571-7598b61528d8-0',
            tool_calls=[],
            invalid_tool_calls=[],
 